In [16]:
import pandas as pd
import numpy as np
import os
from tqdm.notebook import tqdm

os.environ["KERAS_BACKEND"] = "tensorflow"

import numpy as np
from sklearn.preprocessing import normalize

# Prepare the data

In [3]:
game_folder = "data/game_data.csv"
time_folder = "data/time_data.csv"

game_data = pd.read_csv(game_folder)
time_data = pd.read_csv(time_folder)

In [4]:
# make the playtime in hours, remove games with less than 1 hour of playtime
time_data['playtime'] = time_data['playtime'] // 60
time_data = time_data[time_data['playtime'] != 0]
time_data.head()

,user_id,game_id,playtime
0,1,1,1346
1,1,2,10
2,1,3,8
3,1,4,10
4,1,5,6


In [5]:
game_time_sum = time_data.groupby('game_id')['user_id'].count()
game_time_sum = game_time_sum.reset_index()
game_time_sum.columns = ['game_id', 'users']
game_time_sum = game_time_sum.sort_values(by='users', ascending=False)
# get ids of games with at least 50 users, and filter the time_data
game_time_sum = game_time_sum[game_time_sum['users'] >= 50]
game_time_sum = game_time_sum['game_id'].values
time_data = time_data[time_data['game_id'].isin(game_time_sum)]
time_data.shape

(1123490, 3)

In [6]:
# get rid of game_ids that are called unknown_game in the game_data
game_data_ids = game_data[game_data['game_name'] != 'unknown game']["game_id"].values
# filter the time_data
time_data = time_data[time_data['game_id'].isin(game_data_ids)]
time_data.shape

(1079006, 3)

In [7]:
# map user_id and game_id to a unique integer
user_ids = time_data["user_id"].unique().tolist()
user2user_encoded = {x: i for i, x in enumerate(user_ids)}
userencoded2user = {i: x for i, x in enumerate(user_ids)}
game_ids = time_data["game_id"].unique().tolist()
game2game_encoded = {x: i for i, x in enumerate(game_ids)}
game_encoded2game = {i: x for i, x in enumerate(game_ids)}
time_data["user_id"] = time_data["user_id"].map(user2user_encoded)
time_data["game_id"] = time_data["game_id"].map(game2game_encoded)

# get the number of users and games
num_users = len(user2user_encoded)
num_games = len(game_encoded2game)

# check the stats of the data
time_data["playtime"] = time_data["playtime"].values.astype(np.float32)
min_playtime = min(time_data["playtime"])
max_playtime = max(time_data["playtime"])

print(
    f"Number of users: {num_users}, Number of Games: {num_games}, Min playtime: {min_playtime}, Max playtime: {max_playtime}"
)


Number of users: 6560, Number of Games: 6113, Min playtime: 1.0, Max playtime: 75592.0


In [8]:
time_data = time_data.sample(frac=1, random_state=42)
x = time_data[["user_id", "game_id"]].values
y = time_data["playtime"].values

# 90/10 split
train_indices = int(0.9 * time_data.shape[0])
x_train, x_val, y_train, y_val = (
    x[:train_indices],
    x[train_indices:],
    y[:train_indices],
    y[train_indices:],
)

In [9]:
def create_matrix(coordinate_list, values):
    if len(coordinate_list) != len(values):
        raise ValueError("Coordinate list and values should have the same length")
    num_users = max([coordinate[0] for coordinate in coordinate_list]) + 1
    num_games = max([coordinate[1] for coordinate in coordinate_list]) + 1
    matrix = np.zeros((num_users, num_games))
    print(num_users, num_games)
    for i in range(len(coordinate_list)):
        user_id = coordinate_list[i][0]
        game_id = coordinate_list[i][1]
        matrix[user_id, game_id] = values[i]
    return matrix


In [10]:
matrix = create_matrix(x, y)

6560 6113


In [11]:
train_matrix = create_matrix(x_train, y_train)

6560 6113


In [12]:
normalized_matrix = normalize(matrix, axis=1, norm='l2')  # L2 normalization
normalized_train_matrix = normalize(train_matrix, axis=1, norm='l2')  # L2 normalization

# Generate recommendations

In [ ]:
def recommend_for_user(new_user_vector, user_game_matrix,
                       num_neighbors=5, games=20):
    similarities = np.dot(user_game_matrix, new_user_vector)
    similar_user_list = np.argsort(similarities)
    # if the score is higher than 0.999, it means the user is the same as the new user
    # so we will remove this user from the list
    similar_user_list = similar_user_list[similarities[similar_user_list] < 0.999]
    top_similar_users = similar_user_list[-num_neighbors:]

    # calc total playtime of each game by all top_similar_users
    game_scores = np.sum(user_game_matrix[top_similar_users], axis=0)

    # remove games that the new user has already played
    new_user_played_games = np.where(new_user_vector > 0)[0]
    game_scores[new_user_played_games] = 0

    # get top k longest playtime games
    recommended_games = np.argsort(game_scores)[-games:][::-1]

    return recommended_games[:games]

random_user_id = np.random.randint(num_users)

new_user_vector = normalized_matrix[random_user_id]

recommendations = recommend_for_user(new_user_vector, normalized_matrix)

In [14]:
recommendations

array([2775, 5695, 5698, 5701, 5690, 2752, 6055, 2535, 4535, 1283,  285,
       4543,  934, 3260, 2604, 2639, 3116, 2978,  711, 1321])

# Evaluate the model

In [15]:
# check how many unique users in x_test
unique_users = np.unique(x_val[:, 0])
avg_games_per_user = len(x_val) / len(unique_users)
print("Average games per user in validation set:", avg_games_per_user)

Average games per user in validation set: 18.416282642089094


In [188]:
rec_tally = 0


top_neighbor = 15

for id in tqdm(unique_users, total=len(unique_users)):
    # encoded_user_id = user2user_encoded[id]
    new_user_vector = normalized_train_matrix[id]
    recommendations = recommend_for_user(new_user_vector, normalized_train_matrix, top_neighbor)

    # check how many recommendations in x_test where the user_id is equal to id
    user_test = x_val[np.where(x_val[:, 0] == id)][:, 1]
    for game_id in recommendations:
        if game_id in user_test:
            rec_tally += 1

print(f"average games recomennded per user: {rec_tally / len(unique_users)}")

# recommendations = [game_encoded2game.get(game_id) for game_id in recommendations_encoded]


  0%|          | 0/5877 [00:00<?, ?it/s]

average games recomennded per user: 1.0672111621575633


# Generate recommendations for a random user

In [17]:
random_user_id = np.random.randint(num_users)

user_vector = normalized_matrix[random_user_id]

In [19]:
def funny_divider(extra_funny=False):
    print("-^-._." * 6 if extra_funny else "------" * 6)

random_user_id = np.random.randint(num_users)

user_vector = normalized_matrix[random_user_id]

recommendations = recommend_for_user(user_vector, normalized_matrix, num_neighbors=15)
recommendations = recommendations[:20]

funny_divider(True)
print(f"Showing recommendations for user {userencoded2user.get(random_user_id)}")
funny_divider()

# --------- GET TOP GAMES PLAYED BY USER ---------

games_played_by_user = time_data[time_data["user_id"] == random_user_id]
top_games_user = (
    games_played_by_user.sort_values(by="playtime", ascending=False)
    .head(10)
)
top_games_user['game_name'] = top_games_user["game_id"].apply(lambda x: game_data.loc[game_data["game_id"] == game_encoded2game.get(x), "game_name"].values[0])
top_game_ids_playtime = [
    (game_encoded2game.get(x),
    top_games_user.loc[top_games_user["game_id"] == x, "playtime"].values[0]) 
    for x in top_games_user["game_id"]
]

print("Top games played by user:")

funny_divider()


for row in top_games_user.itertuples():
    if row.playtime >= 100:
        print(f"{row.game_name}: >100 hours")
    else:
        print(f"{row.game_name}: {row.playtime} hours")

funny_divider()

# --------- GET RECOMMENDATIONS ---------
# get recommendation ids and extract the game names
recommendation_game_ids = [
    game_encoded2game.get(x) for x in recommendations
]
recommendation_game_names = [
    game_data.loc[game_data["game_id"] == x, "game_name"].values[0]
    for x in recommendation_game_ids
]
print("Recommended games:")

funny_divider()


for game_name in recommendation_game_names:
    print(game_name)

funny_divider(True)

-^-._.-^-._.-^-._.-^-._.-^-._.-^-._.
Showing recommendations for user 4972
------------------------------------
Top games played by user:
------------------------------------
Path of Exile: >100 hours
Rust: >100 hours
Counter-Strike 2: >100 hours
Tom Clancy's Rainbow Six Siege: >100 hours
Crusaders of the Lost Idols: 52.0 hours
PUBG: BATTLEGROUNDS: 28.0 hours
Terraria: 22.0 hours
Once Human: 20.0 hours
Torchlight: Infinite: 16.0 hours
LIV: 15.0 hours
------------------------------------
Recommended games:
------------------------------------
Apex Legends
Last Epoch
AdVenture Capitalist
Street Fighter IV
Phasmophobia
Dungeon Defenders
Warframe
Risk of Rain 2
Sid Meier's Civilization V
The Witcher 3: Wild Hunt
Grim Dawn
Baldur's Gate 3
Wallpaper Engine
Call of Duty: Black Ops III
Sid Meier's Civilization VI
FINAL FANTASY XV WINDOWS EDITION
Crusader Kings II
Fallout: New Vegas
Dead Cells
The Witcher 2: Assassins of Kings Enhanced Edition
-^-._.-^-._.-^-._.-^-._.-^-._.-^-._.
